In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import os

## sensor_timeslot_hist.csv 생성 (약 15,000개 엑셀 파일 합치기)

# 1. 경로 설정
BASE_DIR = Path(os.getcwd()).parent
RAW_DIR = BASE_DIR / "01_data" / "01_raw"
PROCESSED_DIR = BASE_DIR / "01_data" / "02_processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def parse_one_excel(path: Path):
    try:
        # 엑셀 전체 로드
        df = pd.read_excel(path, header=None, engine='openpyxl')
        
        # 1. 파일명에서 메타데이터 추출 (안전한 split)
        # 예: 20241003_원주시_15018524.xlsx
        stem = path.stem
        parts = stem.split('_', 2)
        
        date = parts[0]
        site_name = parts[1] if len(parts) > 1 else "unknown"
        sensor_id = parts[2] if len(parts) > 2 else "unknown"
        
        # 2. 시간대 행 필터링 ("~" 포함된 행)
        mask = df[0].astype(str).str.contains("~", na=False)
        hist_rows = df[mask].copy()
        
        if hist_rows.empty:
            return pd.DataFrame()
            
        data_list = []
        db_cols = [f"db_{i}" for i in range(100)]
        
        for _, row in hist_rows.iterrows():
            new_entry = {
                'date': date,
                'site_name': site_name,
                'sensor_id': sensor_id,
                'timeslot': row[0]  # 0번 열: "02:00 ~ 02:20"
            }
            
            # 1번 열부터 100개(101번 전까지)가 db_0 ~ db_99
            vals = pd.to_numeric(row.iloc[1:101], errors='coerce').fillna(0).astype(int).values
            
            # 데이터 매핑
            for i, col_name in enumerate(db_cols):
                new_entry[col_name] = vals[i]
                
            data_list.append(new_entry)
            
        return pd.DataFrame(data_list)
    
    except Exception as e:
        # print(f"Error in {path.name}: {e}")
        return pd.DataFrame()

# --- 실행 ---
files = [f for f in RAW_DIR.glob("*.xlsx") if not f.name.startswith("~$")]
print(f"🔎 총 {len(files):,}개 파일 처리 시작...")

all_dfs = []
for f in tqdm(files):
    single_df = parse_one_excel(f)
    if not single_df.empty:
        all_dfs.append(single_df)

if all_dfs:
    print("\n데이터 병합 및 저장 중...")
    final_hist = pd.concat(all_dfs, ignore_index=True)
    out_path = PROCESSED_DIR / "01_sensor_timeslot_hist.csv"
    final_hist.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"✅ 저장 완료: {out_path} ({len(final_hist):,}행)")
else:
    print("\n❌ 데이터를 추출하지 못했습니다. 로직을 다시 확인하세요.")


🔎 총 15,833개 파일 처리 시작...


100%|██████████| 15833/15833 [03:18<00:00, 79.87it/s]



데이터 병합 및 저장 중...
✅ 저장 완료: c:\workspace\wvr\classification\labeling_project\01_data\02_processed\sensor_timeslot_hist.csv (94,998행)
